In [2]:
from os import rename

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import importlib
import config
importlib.reload(config)
import pandas_market_calendars as mcal
from scripts.remove_non_trading_days import remove_non_trading_days

from config import (US_TICKERS_FINAL,
                    US_TICKERS_FINAL_adj,
                    BB_PRICES,
                    BB_PRICES_V2,
                    BB_PRICES_COMPLETE)



# This notebook preps tickers to send to Bloomberg to get historical prices.  
# We then load the pricing results and filter the pdf (price dataframe) to:  
 1. remove non trading days.  
 2. remove leading gaps in pricing. i.e. nans before a stock began trading.  
 3. only ffil by 3 days to cover small data gaps. Anything larger than this gets removed.  
4.  Sets a threhold for prices if they dip below $1 and never recover. We remove the bad rows.


In [15]:
#Load tickers file
df = pd.read_csv(US_TICKERS_FINAL)
df.shape
print(df.head())

           ISIN Ticker  Start_Date    End_Date    min_date    max_date  \
0  US8110544025    SSP  05/01/2018  13/06/2025  02/01/2018  28/03/2024   
1  US3020811044   EXLS  23/01/2018  24/06/2025  02/01/2018  28/03/2024   
2  US46269C1027   IRDM  03/01/2018  17/06/2025  02/01/2018  28/03/2024   
3  US63001N1063   NATL  29/01/2024  30/06/2025  02/01/2024  28/03/2024   
4  US7580754023    RWT  20/01/2018  12/06/2025  25/07/2018  28/03/2024   

                   Sector   bb_tcm alt_bb_tcm   mkt_cap  turnover bb_tcm_note  \
0  Consumer Discretionary   SSP US        NaN  1.254711  5.021609         NaN   
1  Information Technology  EXLS US        NaN  2.143838  5.050401         NaN   
2  Communication Services  IRDM US        NaN  1.151158  5.090490         NaN   
3              Financials  NATL US        NaN  1.613916  5.102278         NaN   
4              Financials   RWT US        NaN  1.146697  5.118686         NaN   

  mkt_cap_bin turnover_bin  
0       1–1.5          5-6  
1       2–

In [16]:
df['Start_Date'] = pd.to_datetime(df['Start_Date'],format='%d/%m/%Y')
df['End_Date'] = pd.to_datetime(df['End_Date'],format='%d/%m/%Y')
df['min_date'] = pd.to_datetime(df['min_date'],format='%d/%m/%Y')
df['max_date'] = pd.to_datetime(df['max_date'],format='%d/%m/%Y')

df.rename(columns={'Start_Date':'start_date_s'}, inplace=True)
df.rename(columns={'End_Date':'end_date_s'}, inplace=True)
df.rename(columns={'min_date':'start_date_px'}, inplace=True)
df.rename(columns={'max_date':'end_date_px'}, inplace=True)

print(df.dtypes)

ISIN                     object
Ticker                   object
start_date_s     datetime64[ns]
end_date_s       datetime64[ns]
start_date_px    datetime64[ns]
end_date_px      datetime64[ns]
Sector                   object
bb_tcm                   object
alt_bb_tcm               object
mkt_cap                 float64
turnover                float64
bb_tcm_note              object
mkt_cap_bin              object
turnover_bin             object
dtype: object


In [17]:
df.drop(columns=['mkt_cap_bin','turnover_bin'], inplace=True)

In [5]:
#create the ticker, from_date and to_date list to pass into BQNT
#pd.tseries.offsets.BDay(1) # adds or minus 1 business day on a datetime object. Might be useful.
df['bb_start_date'] = df['start_date_s']-pd.tseries.offsets.BDay(20)
df['bb_end_date'] = df['end_date_s']+pd.tseries.offsets.BDay(20)

In [27]:
df.to_csv(US_TICKERS_FINAL_adj, index=False)

In [8]:
#Load the prices pulled from Bloomberg...
pdf = pd.read_csv(BB_PRICES_V2, sep='\t', low_memory=False, dtype={'ISIN': str})
print(f'pdf shape on load: {pdf.shape}')

# drop any rows where px_last literally contains the string 'px_last'
pdf = pdf[pdf['px_last'] != 'px_last'].copy()

pdf['px_last'] = pdf['px_last'].astype(float)
pdf['date'] = pd.to_datetime(pdf['date'])
pdf = pdf.sort_values(['ISIN', 'date']).reset_index(drop=True)
pdf.drop(columns=['Unnamed: 0','row_id'], inplace=True)

#Run the price frame through the non trading day filter
pdf = remove_non_trading_days(pdf, 'date')

pdf shape on load: (3540035, 7)
There are currently 2794 days in your df
There are 1921 days in the NYSE schedule
Therefore we remove 873 days from the df
df shape after cleaning:(2433754, 5)


In [3]:
pdf.columns

Index(['ISIN', 'Ticker', 'resolved_ticker', 'date', 'px_last'], dtype='object')

In [14]:
print(pdf[(pdf['resolved_ticker']=='SBNY US')  & (pdf['date']>='2023-03-01') & (pdf['date']<='2023-05-01')])

                 ISIN Ticker resolved_ticker       date    px_last
1969747  US82669G1040   SBNY         SBNY US 2023-03-01  112.61000
1969748  US82669G1040   SBNY         SBNY US 2023-03-02  109.56000
1969749  US82669G1040   SBNY         SBNY US 2023-03-03  113.70000
1969750  US82669G1040   SBNY         SBNY US 2023-03-06  110.89000
1969751  US82669G1040   SBNY         SBNY US 2023-03-07  104.89000
1969752  US82669G1040   SBNY         SBNY US 2023-03-08  103.35000
1969753  US82669G1040   SBNY         SBNY US 2023-03-09   90.76000
1969754  US82669G1040   SBNY         SBNY US 2023-03-10   70.00000
1969755  US82669G1040   SBNY         SBNY US 2023-03-13   70.00000
1969756  US82669G1040   SBNY         SBNY US 2023-03-14   70.00000
1969757  US82669G1040   SBNY         SBNY US 2023-03-15   70.00000
1969758  US82669G1040   SBNY         SBNY US 2023-03-28    0.13000
1969759  US82669G1040   SBNY         SBNY US 2023-03-29    0.24000
1969760  US82669G1040   SBNY         SBNY US 2023-03-30    0.1

In [ ]:

# 1. Diagnose before touching anything - know what you're about to remove
lead_nan = (pdf.groupby('ISIN')['px_last']
            .apply(lambda s: s.isna().cumprod().sum()))   # leading-NaN run length
print(f"Securities with leading NaNs: {(lead_nan > 0).sum()}")
print(f"Total leading-NaN rows:       {int(lead_nan.sum()):,}")
print(f"Interior/other NaNs:          {int(pdf['px_last'].isna().sum() - lead_nan.sum()):,}")

# 2. Drop leading gaps - keep each stock only from its first real price - explain why you didn't use backfill.
first_valid = (pdf.dropna(subset=['px_last'])
               .groupby('ISIN')['date'].min()
               .rename('first_px_date'))
pdf = pdf.merge(first_valid, on='ISIN', how='inner')
pdf = pdf[pdf['date'] >= pdf['first_px_date']].drop(columns='first_px_date')

# 3. Bridge short interior gaps only - this covers small trading halts. Anything longer needs to be dropped on the next step.
pdf['px_last'] = pdf.groupby('ISIN')['px_last'].ffill(limit=3)

# 4. Drop what refused to fill - halts and long absences
pdf = pdf.dropna(subset=['px_last'])

In [ ]:

# 5. Price threshold of USD 1.00. If a stock drops below this price then we cut the rest of those observations as the stock
# is no longer tradeable.
THRESH = 1.00
last_valid = (pdf[pdf['px_last'] >= THRESH]
              .groupby('ISIN')['date'].max()
              .rename('last_valid_date'))
pdf = pdf.merge(last_valid, on='ISIN', how='inner')
pdf = pdf[pdf['date'] <= pdf['last_valid_date']].drop(columns='last_valid_date')

In [13]:
# 6. Export
pdf.to_parquet(BB_PRICES_COMPLETE,index=False)